<a href="https://colab.research.google.com/github/sAI-2025/Jiohotstar_recomendationAgent/blob/main/Jiohostor_csvTOsql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CSV to SQL

In [77]:
import pandas as pd
df = pd.read_csv('/content/Jiohostar_imbd_data.csv')

In [78]:
df.head(2)

,title,released_year,duration,genre,description,Rated,Released,Runtime,Genre,Director,...,imdbID,Type,DVD,BoxOffice,Production,Website,api_error,totalSeasons,imageid,format
0,Waqt,1965,2h 58m,Drama,"Separated in childhood, three brothers come to...",NaN,28 Jul 1965,178 min,"Drama, Romance",Yash Chopra,...,tt0059893,movie,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN
1,Badrinath Yatra,1967,2h,Crime,Satyavati's pilgrim to Badrinath gets cancelle...,NaN,01 Jan 1967,140 min,"Crime, Drama",Dhirubhai Desai,...,tt0213476,movie,NaN,NaN,NaN,NaN,NaN,NaN,2,NaN


In [79]:
df.columns.tolist()

['title',
 'released_year',
 'duration',
 'genre',
 'description',
 'Rated',
 'Released',
 'Runtime',
 'Genre',
 'Director',
 'Writer',
 'Actors',
 'Plot',
 'Language',
 'Country',
 'Awards',
 'Poster',
 'Ratings',
 'Metascore',
 'imdbRating',
 'imdbVotes',
 'imdbID',
 'Type',
 'DVD',
 'BoxOffice',
 'Production',
 'Website',
 'api_error',
 'totalSeasons',
 'imageid',
 'format']

In [80]:
df.shape

(461, 31)

In [81]:
df.isnull().sum()


,0
title,0
released_year,0
duration,0
genre,0
description,0
Rated,278
Released,115
Runtime,113
Genre,107
Director,111


In [82]:
# print(df.nunique())

In [83]:
df['Rated'].value_counts()

,count
Rated,
Not Rated,170
TV-MA,4
Unrated,3
TV-14,2
TV-PG,2
NOT RATED,1
PG,1


#### Removing some unnessary columns

In [84]:
df.drop(columns=[
    'DVD',
    'api_error',
    'Website',
    'Production',
    'Metascore',
    'Poster'
], inplace=True)

In [85]:
df.columns

Index(['title', 'released_year', 'duration', 'genre', 'description', 'Rated',
       'Released', 'Runtime', 'Genre', 'Director', 'Writer', 'Actors', 'Plot',
       'Language', 'Country', 'Awards', 'Ratings', 'imdbRating', 'imdbVotes',
       'imdbID', 'Type', 'BoxOffice', 'totalSeasons', 'imageid', 'format'],
      dtype='object')

In [86]:
df.shape

(461, 25)

### `Null` values repaced into the `None`

In [87]:
df = df.where(pd.notna(df), None)

###  `BoxOffice` column  convert `dollar` to `Rupes` ( as like cores rupes)


In [88]:
df.BoxOffice

,BoxOffice
0,None
1,None
2,None
3,None
4,None
...,...
456,None
457,None
458,"$335,775"
459,None


In [89]:
USD_TO_INR = 83  # Exchange rate

def dollar_to_indian(value):
    if pd.isna(value) or value is None:
        return None

    # Remove $ and commas
    usd = float(value.replace('$', '').replace(',', ''))

    # Convert USD to INR
    inr = usd * USD_TO_INR

    if inr >= 10_000_000:  # 1 Crore
        crore = inr / 10_000_000
        return f"₹{crore:.2f} Cr"
    else:
        lakh = round(inr / 100_000)  # Rounded to nearest lakh
        return f"₹{lakh} Lakh"

df['BoxOffice'] = df['BoxOffice'].apply(dollar_to_indian)

In [90]:
df.BoxOffice

,BoxOffice
0,None
1,None
2,None
3,None
4,None
...,...
456,None
457,None
458,₹2.79 Cr
459,None


,count
Genre,
"Action, Crime, Drama",25
"Comedy, Drama, Romance",20
"Action, Drama",19
Drama,19
"Action, Comedy, Drama",16
...,...
"Crime, Drama, Music",1
Sport,1
"Drama, Mystery, Romance",1


### Merging Multiple Columns with Duplicate and Missing Value Handling

This process combines values from two similar columns (`genre` and `Genre`) into a single column while maintaining clean and consistent data. The values from both columns are merged, separated by commas, and processed to remove duplicate entries, extra spaces, empty values, and missing values such as `None` or `NaN`. The original order of unique values is preserved to keep the data readable and meaningful. After merging, the unnecessary duplicate column (`Genre`) is removed from the DataFrame.

**Steps performed:**
- Combine values from `genre` and `Genre` columns.
- Split comma-separated values into individual items.
- Remove leading and trailing spaces.
- Ignore missing values (`None`/`NaN`).
- Remove duplicate values while preserving order.
- Store the final cleaned values in the `genre` column.
- Delete the redundant `Genre` column.

In [93]:
df[["genre","Genre"]].head(5)

,genre,Genre
0,Drama,"Drama, Romance"
1,Crime,"Crime, Drama"
2,Action,"Action, Drama"
3,Comedy,"Comedy, Romance"
4,Drama,"Drama, Family"


In [95]:
def merge_genres(row):
    values = []

    for col in ['genre', 'Genre']:
        value = row[col]

        # Ignore None / NaN values
        if value is not None and str(value).lower() != 'none':
            values.extend(str(value).split(','))

    # Remove spaces, remove empty values
    values = [x.strip() for x in values if x.strip()]

    # Remove duplicates while keeping order
    values = list(dict.fromkeys(values))

    return ', '.join(values)

In [96]:
df['genre'] = df.apply(merge_genres, axis=1)

In [97]:
df[["genre","Genre"]].head(5)

,genre,Genre
0,"Drama, Romance","Drama, Romance"
1,"Crime, Drama","Crime, Drama"
2,"Action, Drama","Action, Drama"
3,"Comedy, Romance","Comedy, Romance"
4,"Drama, Family","Drama, Family"


In [98]:
df.drop(columns=['Genre'], inplace=True)

### Splitting Rows with Multiple Field Values into Separate Rows

This process converts comma-separated values in multiple columns into individual rows. Each column (such as Language, Country, and Genre) is split separately by commas, converted into a list of values, and expanded using `explode()`. Since columns are exploded one by one, the number of rows increases, and it may create combinations of values between different fields. This approach is useful when analyzing each value separately, but it may not always preserve the original relationship between values across columns.

**Example Columns:**
- Language
- Country
- Genre

In [99]:
# Columns to split
cols = ['Language', 'Country', 'genre']

for col in cols:
    df[col] = df[col].apply(
        lambda x: [i.strip() for i in x.split(',')] if isinstance(x, str) else x
    )
    df = df.explode(col, ignore_index=True)

In [103]:
df.head(4)

,title,released_year,duration,genre,description,Rated,Released,Runtime,Director,Writer,...,Awards,Ratings,imdbRating,imdbVotes,imdbID,Type,BoxOffice,totalSeasons,imageid,format
0,Waqt,1965,2h 58m,Drama,"Separated in childhood, three brothers come to...",None,28 Jul 1965,178 min,Yash Chopra,"Akhtar Mirza, Akhtar-Ul-Iman",...,5 wins & 2 nominations,"[{""Source"": ""Internet Movie Database"", ""Value""...",7.6,"1,555",tt0059893,movie,None,NaN,1,None
1,Waqt,1965,2h 58m,Romance,"Separated in childhood, three brothers come to...",None,28 Jul 1965,178 min,Yash Chopra,"Akhtar Mirza, Akhtar-Ul-Iman",...,5 wins & 2 nominations,"[{""Source"": ""Internet Movie Database"", ""Value""...",7.6,"1,555",tt0059893,movie,None,NaN,1,None
2,Badrinath Yatra,1967,2h,Crime,Satyavati's pilgrim to Badrinath gets cancelle...,None,01 Jan 1967,140 min,Dhirubhai Desai,"Dhirubhai Desai, Pandit Raj Yogi",...,None,[],NaN,11,tt0213476,movie,None,NaN,2,None
3,Badrinath Yatra,1967,2h,Drama,Satyavati's pilgrim to Badrinath gets cancelle...,None,01 Jan 1967,140 min,Dhirubhai Desai,"Dhirubhai Desai, Pandit Raj Yogi",...,None,[],NaN,11,tt0213476,movie,None,NaN,2,None


In [102]:
df.columns

Index(['title', 'released_year', 'duration', 'genre', 'description', 'Rated',
       'Released', 'Runtime', 'Director', 'Writer', 'Actors', 'Plot',
       'Language', 'Country', 'Awards', 'Ratings', 'imdbRating', 'imdbVotes',
       'imdbID', 'Type', 'BoxOffice', 'totalSeasons', 'imageid', 'format'],
      dtype='object')

In [101]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


In [107]:
print(df['Rated'].value_counts())

Rated
Not Rated    568
TV-MA         17
Unrated       15
TV-PG         12
TV-14          5
NOT RATED      2
PG             2
Name: count, dtype: int64


In [108]:
df.shape

(1142, 24)

In [112]:
df.isnull().sum()

,0
title,0
released_year,0
duration,0
genre,0
description,0
Rated,521
Released,124
Runtime,116
Director,125
Writer,161


In [116]:
df.dtypes

,0
title,object
released_year,int64
duration,object
genre,object
description,object
Rated,object
Released,object
Runtime,object
Director,object
Writer,object


In [117]:
df.to_csv("movies_sqlite_db.csv", index=False)

In [118]:
import sqlite3

# Create SQLite database connection
conn = sqlite3.connect("movies_sqlite_db.db")

# Save DataFrame as a table
df.to_sql("movies", conn, if_exists="replace", index=False)

# Close connection
conn.close()